In [44]:
from pyspark.sql import SparkSession
from collections import Counter
from pyspark.sql.functions import sum, desc, countDistinct, year, month, avg, datediff, round, dense_rank
from pyspark.sql.window import Window
import os
import builtins



In [45]:
spark = (
    SparkSession.builder
    .appName("TradeCorpTransformations")
    .getOrCreate()
)

PATH = "/home/jovyan/data/tmp"

In [46]:
df_categories = spark.read.parquet(f"{PATH}/categories.parquet")
df_customers = spark.read.parquet(f"{PATH}/customers.parquet")
df_employees = spark.read.parquet(f"{PATH}/employees.parquet")
df_order_details = spark.read.parquet(f"{PATH}/order_details.parquet")
df_orders = spark.read.parquet(f"{PATH}/orders.parquet")
df_products = spark.read.parquet(f"{PATH}/products.parquet")
df_shippers = spark.read.parquet(f"{PATH}/shippers.parquet")
df_suppliers = spark.read.parquet(f"{PATH}/suppliers.parquet")

## Q21.

In [47]:
df_orders_customers = (
    df_orders
    .join(df_customers, on="customer_id", how="left")
    .select(
        "order_id",
        "company_name",
        "country",
        "order_date",
        "freight"
    )
)

## Q22.

In [48]:
df_order_details_products = (
    df_order_details
    .join(
        df_products.select(
            "product_id",
            "product_name",
            "category_id",
            "unit_price"
        ),
        on="product_id",
        how="left"
    )
)

## Q23.

In [49]:
df_products_categories = (
    df_products
    .join(
        df_categories.select(
            "category_id",
            "category_name",
            "description"
        ),
        on="category_id",
        how="left"
    )
)

## Q24A.

In [50]:
df_orders_enrichi_a = (
    df_order_details
    .join(df_orders, on="order_id", how="left")
    .join(df_customers, on="customer_id", how="left")
    .join(df_products_categories, on="product_id", how="left")
    .join(df_employees, on="employee_id", how="left")
    .join(df_shippers, on="shipper_id", how="left")
)

In [51]:
column_counts = Counter(df_orders_enrichi_a.columns)

duplicates = [
    column
    for column, count in column_counts.items()
    if count > 1
]

print("Colonnes en double :", duplicates)

Colonnes en double : ['company_name', 'city', 'country', 'phone']


## Q24B.

In [52]:
df_customers_renomme = (
    df_customers
    .withColumnRenamed("company_name", "customer_company_name")
    .withColumnRenamed("city", "customer_city")
    .withColumnRenamed("country", "customer_country")
    .withColumnRenamed("phone", "customer_phone")
)

df_employees_renomme = (
    df_employees
    .withColumnRenamed("city", "employee_city")
    .withColumnRenamed("country", "employee_country")
)

df_shippers_renomme = (
    df_shippers
    .withColumnRenamed("company_name", "shipper_name")
    .withColumnRenamed("phone", "shipper_phone")
)

In [53]:
df_orders_enriched = (
    df_order_details
    .join(df_orders, on="order_id", how="left")
    .join(df_customers_renomme, on="customer_id", how="left")
    .join(df_products_categories, on="product_id", how="left")
    .join(df_employees_renomme, on="employee_id", how="left")
    .join(df_shippers_renomme, on="shipper_id", how="left")
)

In [54]:
column_counts = Counter(df_orders_enriched.columns)

duplicates = [
    column
    for column, count in column_counts.items()
    if count > 1
]

print("Colonnes encore en double :", duplicates)

Colonnes encore en double : []


## Q25.

In [55]:
df_ca_clients = (
    df_orders_enriched
    .groupBy("customer_company_name")
    .agg(
        sum("sous_total").alias("ca_total")
    )
    .orderBy(desc("ca_total"))
)

df_ca_clients.show(10, truncate=False)

+----------------------------+------------------+
|customer_company_name       |ca_total          |
+----------------------------+------------------+
|NULL                        |648707.9100000003 |
|QUICK-Stop                  |61109.92          |
|Save-a-lot Markets          |57713.58000000001 |
|Ernst Handel                |48096.27          |
|Mère Paillarde              |23332.320000000003|
|Hungry Owl All-Night Grocers|20454.41          |
|Rattlesnake Canyon Grocery  |19383.75          |
|Simons bistro               |16232.42          |
|Berglunds snabbköp          |13849.02          |
|HILARION-Abastos            |13482.740000000002|
+----------------------------+------------------+
only showing top 10 rows


## Q26.

In [56]:
df_ca_categories = (
    df_orders_enriched
    .groupBy("category_name")
    .agg(
        sum("sous_total").alias("ca_total"),
        countDistinct("product_id").alias("nb_produits_distincts")
    )
    .orderBy(desc("ca_total"))
)

df_ca_categories.show(truncate=False)

+--------------+------------------+---------------------+
|category_name |ca_total          |nb_produits_distincts|
+--------------+------------------+---------------------+
|Beverages     |234219.77         |9                    |
|NULL          |229055.91         |11                   |
|Dairy Products|219586.43000000002|9                    |
|Confections   |167357.26         |13                   |
|Seafood       |131261.75999999998|12                   |
|Condiments    |100699.94         |11                   |
|Grains/Cereals|87169.59999999999 |6                    |
|Produce       |74287.94          |4                    |
|Meat/Poultry  |22154.640000000003|2                    |
+--------------+------------------+---------------------+



## Q27.

In [57]:
df_ca_mensuel = (
    df_orders_enriched
    .groupBy(
        year("order_date").alias("annee"),
        month("order_date").alias("mois")
    )
    .agg(
        sum("sous_total").alias("ca_total")
    )
    .orderBy("annee", "mois")
)

df_ca_mensuel.show(truncate=False)

+-----+----+------------------+
|annee|mois|ca_total          |
+-----+----+------------------+
|NULL |NULL|648707.9100000003 |
|1997 |1   |61258.08          |
|1997 |2   |38483.64          |
|1997 |3   |38547.229999999996|
|1997 |4   |53032.95          |
|1997 |5   |53781.3           |
|1997 |6   |36362.82          |
|1997 |7   |51020.88          |
|1997 |8   |47287.68          |
|1997 |9   |55629.27          |
|1997 |10  |66749.24000000002 |
|1997 |11  |43533.80999999999 |
|1997 |12  |71398.44          |
+-----+----+------------------+



## Q28.

In [58]:
df_performance_employes = (
    df_orders_enriched
    .groupBy("full_name")
    .agg(
        countDistinct("order_id").alias("nb_commandes"),
        round(sum("sous_total"), 2).alias("ca_total"),
        round(
            avg(datediff("shipped_date", "order_date")),
            2
        ).alias("delai_moyen_livraison_jours")
    )
)

df_performance_employes.show(truncate=False)

+----------------+------------+---------+---------------------------+
|full_name       |nb_commandes|ca_total |delai_moyen_livraison_jours|
+----------------+------------+---------+---------------------------+
|NULL            |422         |648707.91|NULL                       |
|Anne Dodsworth  |19          |26310.39 |11.51                      |
|Nancy Davolio   |55          |93148.12 |7.58                       |
|Andrew Fuller   |41          |70444.14 |10.5                       |
|Steven Buchanan |18          |30716.49 |6.6                        |
|Janet Leverling |71          |108026.17|8.84                       |
|Robert King     |36          |60471.19 |9.52                       |
|Laura Callahan  |54          |56032.63 |7.85                       |
|Margaret Peacock|81          |128809.83|8.16                       |
|Michael Suyama  |33          |43126.38 |8.38                       |
+----------------+------------+---------+---------------------------+



## Q29.

In [59]:
# Calcul du CA par produit et par catégorie
df_ca_produits = (
    df_orders_enriched
    .groupBy(
        "category_name",
        "product_id",
        "product_name"
    )
    .agg(
        sum("sous_total").alias("ca")
    )
)

# Fenêtre partitionnée par catégorie
window_category = (
    Window
    .partitionBy("category_name")
    .orderBy(desc("ca"))
)

# Classement des produits dans chaque catégorie
df_ranking_produits = (
    df_ca_produits
    .withColumn(
        "rang",
        dense_rank().over(window_category)
    )
    .orderBy("category_name", "rang")
)

df_ranking_produits.show(truncate=False)

+-------------+----------+-------------------------+------------------+----+
|category_name|product_id|product_name             |ca                |rang|
+-------------+----------+-------------------------+------------------+----+
|NULL         |29        |NULL                     |80368.69000000002 |1   |
|NULL         |17        |NULL                     |32698.379999999997|2   |
|NULL         |28        |NULL                     |25696.640000000007|3   |
|NULL         |53        |NULL                     |20574.170000000006|4   |
|NULL         |2         |NULL                     |16355.96          |5   |
|NULL         |31        |NULL                     |14920.89          |6   |
|NULL         |1         |NULL                     |12788.1           |7   |
|NULL         |42        |NULL                     |8575.0            |8   |
|NULL         |9         |NULL                     |7226.5            |9   |
|NULL         |5         |NULL                     |5347.21           |10  |

## Q30.

In [60]:
# CA par mois
df_ca_mensuel = (
    df_orders_enriched
    .groupBy(
        year("order_date").alias("annee"),
        month("order_date").alias("mois")
    )
    .agg(
        sum("sous_total").alias("ca_mensuel")
    )
)

# Fenêtre 
window_cumul = (
    Window
    .orderBy("annee", "mois")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# CA cumulé
df_ca_cumule = (
    df_ca_mensuel
    .withColumn(
        "ca_cumule",
        sum("ca_mensuel").over(window_cumul)
    )
    .orderBy("annee", "mois")
)

df_ca_cumule.show(truncate=False)

+-----+----+------------------+------------------+
|annee|mois|ca_mensuel        |ca_cumule         |
+-----+----+------------------+------------------+
|NULL |NULL|648707.9100000003 |648707.9100000003 |
|1997 |1   |61258.08          |709965.9900000002 |
|1997 |2   |38483.64          |748449.6300000002 |
|1997 |3   |38547.229999999996|786996.8600000002 |
|1997 |4   |53032.95          |840029.8100000002 |
|1997 |5   |53781.3           |893811.1100000002 |
|1997 |6   |36362.82          |930173.9300000002 |
|1997 |7   |51020.88          |981194.8100000002 |
|1997 |8   |47287.68          |1028482.4900000002|
|1997 |9   |55629.27          |1084111.7600000002|
|1997 |10  |66749.24000000002 |1150861.0000000002|
|1997 |11  |43533.80999999999 |1194394.8100000003|
|1997 |12  |71398.44          |1265793.2500000002|
+-----+----+------------------+------------------+



## Q31.

In [61]:
# 5 produits les plus vendus en quantité
df_top_produits = (
    df_orders_enriched
    .groupBy("product_id", "product_name")
    .agg(
        sum("quantite").alias("quantite_totale")
    )
    .orderBy(desc("quantite_totale"))
    .limit(5)
)

df_top_produits.show(truncate=False)

+----------+----------------------+---------------+
|product_id|product_name          |quantite_totale|
+----------+----------------------+---------------+
|60        |Camembert Pierrot     |1577           |
|59        |Raclette Courdavault  |1496           |
|31        |NULL                  |1397           |
|56        |Gnocchi di nonna Alice|1263           |
|16        |Pavlova               |1158           |
+----------+----------------------+---------------+



In [62]:
# 3 pays clients générant le plus de chiffre d'affaires
df_top_pays = (
    df_orders_enriched
    .groupBy("customer_country")
    .agg(
        sum("sous_total").alias("ca_total")
    )
    .orderBy(desc("ca_total"))
    .limit(3)
)

df_top_pays.show(truncate=False)

+----------------+------------------+
|customer_country|ca_total          |
+----------------+------------------+
|NULL            |648707.9100000003 |
|GERMANY         |117320.20000000001|
|USA             |114845.28999999998|
+----------------+------------------+



## Q32.

In [63]:
df_orders_enriched.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/data/output/orders_enriched.parquet")

## Q33.

In [64]:
# Relire le fichier Parquet
df_orders_enriched_parquet = spark.read.parquet(
    "/home/jovyan/data/output/orders_enriched.parquet"
)

# Comparer le nombre de lignes
nb_original = df_orders_enriched.count()
nb_parquet = df_orders_enriched_parquet.count()

print("Nombre de lignes original :", nb_original)
print("Nombre de lignes Parquet :", nb_parquet)

if nb_original == nb_parquet:
    print("Le nombre de lignes est identique.")
else:
    print("Le nombre de lignes est différent.")

df_orders_enriched_parquet.printSchema()

Nombre de lignes original : 2155
Nombre de lignes Parquet : 2155
Le nombre de lignes est identique.
root
 |-- shipper_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- sous_total: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)
 |-- is_shipped: boolean (nullable = true)
 |-- customer_company_name: string (nullable 

## Q34.

In [65]:
csv_path = "/home/jovyan/data/raw/orders.csv"
parquet_path = "/home/jovyan/data/output/orders_enriched.parquet"

# Taille du CSV
csv_size = os.path.getsize(csv_path)

# Taille totale du dossier Parquet
parquet_size = builtins.sum(
    os.path.getsize(os.path.join(root, file))
    for root, _, files in os.walk(parquet_path)
    for file in files
)

print(f"Taille CSV     : {csv_size / 1024:.2f} Ko")
print(f"Taille Parquet : {parquet_size / 1024:.2f} Ko")

Taille CSV     : 98.76 Ko
Taille Parquet : 95.75 Ko


Le format Parquet est plus efficace que CSV car :

- il utilise un stockage en colonnes
- il applique de la compression 
- il conserve les types de données 
Spark peut lire uniquement les colonnes nécessaires au lieu de parcourir toutes les données 

## Q35.

In [66]:
output_path = "/home/jovyan/data/output/orders_partitioned"

df_orders_enriched.write \
    .mode("overwrite") \
    .partitionBy("customer_country") \
    .parquet(output_path)

In [67]:
# OBSERVATION DES FICHIERS
for root, dirs, files in os.walk(output_path):
    level = root.replace(output_path, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

orders_partitioned/
    customer_country=ARGENTINA/
    customer_country=AUSTRIA/
    customer_country=BELGIUM/
    customer_country=BRAZIL/
    customer_country=CANADA/
    customer_country=DENMARK/
    customer_country=FINLAND/
    customer_country=FRANCE/
    customer_country=GERMANY/
    customer_country=IRELAND/
    customer_country=ITALY/
    customer_country=MEXICO/
    customer_country=NORWAY/
    customer_country=POLAND/
    customer_country=PORTUGAL/
    customer_country=SPAIN/
    customer_country=SWEDEN/
    customer_country=SWITZERLAND/
    customer_country=UK/
    customer_country=USA/
    customer_country=VENEZUELA/
    customer_country=__HIVE_DEFAULT_PARTITION__/


## Q36.

In [68]:
jdbc_url = "jdbc:postgresql://postgres:5432/tradecorp"

properties = {
    "user": "tradecorp",
    "password": "tradecorp",
    "driver": "org.postgresql.Driver"
}

df_orders_enriched.write \
    .mode("overwrite") \
    .jdbc(
        url=jdbc_url,
        table="orders_enriched",
        properties=properties
    )